# POC Snowflake — SwissBike SA

## Objectif

SwissBike SA vend des vélos et accessoires en ligne. L'entreprise grandit et rencontre plusieurs problèmes :

- les données sont dispersées entre l'e-commerce, l'ERP, le CRM et des fichiers Excel ;
- les rapports mensuels prennent du temps ;
- Finance, Marketing et Direction n'ont pas toujours les mêmes chiffres ;
- le volume de commandes augmente ;
- une erreur humaine peut supprimer ou modifier une table importante.

Le but du POC est de montrer comment Snowflake peut aider à répondre à ces problèmes.

## 1. Vérification de l'environnement Snowflake

Cette première requête vérifie le compte, l'utilisateur et le warehouse utilisé.

In [ ]:
%%sql -r dataframe_1
SELECT
    CURRENT_ACCOUNT() AS ACCOUNT,
    CURRENT_USER() AS USER,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

## 2. Données utilisées pour le POC

Pour simuler les données de SwissBike SA, nous utilisons le dataset d'exemple **TPC-H** fourni par Snowflake.

TPC-H représente une entreprise fictive avec :

- des clients ;
- des commandes ;
- des lignes de commandes ;
- des fournisseurs ;
- des pays et régions.

Cela permet de faire une démonstration réaliste sans importer manuellement des fichiers CSV.

In [ ]:
%%sql -r dataframe_2
SHOW TABLES IN SCHEMA SNOWFLAKE_SAMPLE_DATA.TPCH_SF1;

## 3. Volume de données

Avant d'analyser les données, on regarde rapidement la taille des tables principales.

In [ ]:
%%sql -r dataframe_3
SELECT 'CUSTOMER' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
UNION ALL
SELECT 'ORDERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
UNION ALL
SELECT 'LINEITEM' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM;

## 4. Situation métier : quels pays génèrent le plus de chiffre d'affaires ?

La direction de SwissBike SA veut identifier les marchés les plus importants.

Cette requête joint plusieurs tables et agrège le chiffre d'affaires par pays.

In [ ]:
%%sql -r dataframe_4
SELECT
    N.N_NAME AS COUNTRY,
    COUNT(DISTINCT O.O_ORDERKEY) AS NB_ORDERS,
    ROUND(SUM(L.L_EXTENDEDPRICE), 2) AS REVENUE
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM L
JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS O
    ON L.L_ORDERKEY = O.O_ORDERKEY
JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER C
    ON O.O_CUSTKEY = C.C_CUSTKEY
JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION N
    ON C.C_NATIONKEY = N.N_NATIONKEY
GROUP BY N.N_NAME
ORDER BY REVENUE DESC;

Cette requête montre le rôle principal d'un data warehouse : centraliser des données et permettre des analyses business avec SQL.

Dans Snowflake, la requête est exécutée par un **Virtual Warehouse**, c'est-à-dire une ressource de calcul séparée du stockage.

## 5. MPP : Massively Parallel Processing

Pour montrer le MPP :

1. exécuter la requête précédente ;
2. aller dans **History** ;
3. ouvrir la requête ;
4. cliquer sur **Query Profile**.

Snowflake découpe une requête complexe en plusieurs opérations exécutées en parallèle. C'est le principe du MPP. Plus le warehouse est grand, plus Snowflake dispose de ressources de calcul pour paralléliser certaines opérations.

## 6. Scalabilité : changer la taille du warehouse

Snowflake permet de modifier la puissance de calcul sans déplacer les données.

Attention : cette commande peut avoir un impact sur les coûts. Pour un POC, remettre ensuite le warehouse en XSMALL.

In [ ]:
%%sql -r dataframe_5
-- Agrandir temporairement le warehouse
ALTER WAREHOUSE COMPUTE_WH
SET WAREHOUSE_SIZE = 'SMALL';

In [ ]:
%%sql -r dataframe_6
-- Revenir à une taille moins coûteuse après la démo
ALTER WAREHOUSE COMPUTE_WH
SET WAREHOUSE_SIZE = 'XSMALL';

La séparation entre **storage** et **compute** permet d'adapter la puissance aux besoins.

Exemple PME :

- petit warehouse pendant les analyses normales ;
- warehouse plus grand pendant les rapports mensuels ;
- arrêt automatique lorsque personne ne l'utilise.

## 7. Création d'une base de travail pour le POC

On crée une base et un schéma dédiés à la démonstration.

In [ ]:
%%sql -r dataframe_7
CREATE DATABASE IF NOT EXISTS POC_SNOWFLAKE;
CREATE SCHEMA IF NOT EXISTS POC_SNOWFLAKE.DEMO;

USE DATABASE POC_SNOWFLAKE;
USE SCHEMA DEMO;

## 8. Création d'une table de production simulée

On copie les clients dans une table locale pour simuler une table de production de SwissBike SA.

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE CUSTOMER_PROD AS
SELECT *
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER;

In [ ]:
%%sql -r dataframe_9
SELECT COUNT(*) AS NB_CUSTOMERS
FROM CUSTOMER_PROD;

## 9. Time Travel : récupération après une erreur humaine

Situation :

Un analyste supprime accidentellement une table importante.

Dans un système traditionnel, il faudrait souvent restaurer un backup. Avec Snowflake, on peut utiliser **Time Travel**.

In [ ]:
%%sql -r dataframe_10
DROP TABLE CUSTOMER_PROD;

La table a été supprimée. On la restaure avec `UNDROP TABLE`.

In [ ]:
%%sql -r dataframe_11
UNDROP TABLE CUSTOMER_PROD;

In [ ]:
%%sql -r dataframe_12
SELECT COUNT(*) AS NB_CUSTOMERS_AFTER_RESTORE
FROM CUSTOMER_PROD;

Time Travel réduit le risque opérationnel.

Pour une PME, cela signifie :

- récupération plus rapide ;
- moins de dépendance à une restauration manuelle ;
- meilleure sécurité face aux erreurs humaines.

## 10. Zero-Copy Clone : créer un environnement de test

Situation :

L'équipe Data veut tester une nouvelle analyse sans modifier la production.

Snowflake permet de créer un clone presque instantané d'une table, d'un schéma ou d'une base.

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE TABLE CUSTOMER_TEST
CLONE CUSTOMER_PROD;

In [ ]:
%%sql -r dataframe_14
SELECT COUNT(*) AS NB_CUSTOMERS_TEST
FROM CUSTOMER_TEST;

On peut modifier la table de test sans impacter la table de production.

In [ ]:
%%sql -r dataframe_15
DELETE FROM CUSTOMER_TEST
WHERE C_MKTSEGMENT = 'AUTOMOBILE';

In [ ]:
%%sql -r dataframe_16
SELECT 'CUSTOMER_PROD' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM CUSTOMER_PROD
UNION ALL
SELECT 'CUSTOMER_TEST' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM CUSTOMER_TEST;

Le clone permet de créer rapidement un environnement de test.

C'est utile pour :

- tester une transformation ;
- préparer une nouvelle version d'un pipeline ;
- faire de l'analyse exploratoire sans risque.

## 11. Synthèse : besoin PME → réponse Snowflake

| Problème de SwissBike SA | Fonctionnalité Snowflake |
|---|---|
| Données dispersées | Data Warehouse centralisé |
| Rapports analytiques | SQL + moteur analytique |
| Requêtes complexes | MPP |
| Croissance du volume | Scalabilité du warehouse |
| Erreur humaine | Time Travel |
| Environnement de test | Zero-Copy Clone |
| Équipes multiples | Multi-cluster compute |

## Limites à mentionner

Snowflake n'est pas toujours la meilleure solution.

Points critiques :

- coût parfois difficile à prévoir ;
- dépendance au fournisseur ;
- moins adapté aux petites bases de données simples ;
- pas conçu pour remplacer une base transactionnelle OLTP ;
- nécessite une gouvernance des rôles, warehouses et coûts.

## 12. Nettoyage optionnel

À exécuter seulement après la démonstration.

In [ ]:
%%sql -r dataframe_17
-- DROP DATABASE IF EXISTS POC_SNOWFLAKE;